In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_2.py ──
"""
Shared infrastructure for MLFP03 Exercise 2 — Regularisation and
Cross-Validation.

Contains: data loading for the Singapore credit scoring dataset,
feature preparation, synthetic 1D bias-variance problem, alpha grids,
plotting helpers, and a shared OUTPUT_DIR for generated artefacts.

Technique-specific code (Ridge/Lasso model construction, nested CV
loops, learning curves) lives in the per-technique solution files —
this module holds only the helpers those files share.
"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ════════════════════════════════════════════════════════════════════════

# Deterministic seed for every random operation in this exercise
SEED = 42

# Alpha sweep used by Ridge / Lasso / ElasticNet and the regularisation path
ALPHAS: list[float] = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]

# Output directory for HTML plots, summary CSVs, etc.
OUTPUT_DIR = Path("outputs") / "mlfp03_ex2_regularisation_cv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Singapore credit scoring
# ════════════════════════════════════════════════════════════════════════


def load_credit_data() -> (
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, list[str]]
):
    """Load + preprocess the MLFP02 Singapore credit scoring parquet.

    Returns:
        X_train, y_train, X_test, y_test, feature_names

    Target: predicts ``credit_utilization`` (continuous ratio). Falls
    back to ``income_sgd`` if the utilisation column is absent in older
    dataset revisions.

    Uses ``kailash_ml.PreprocessingPipeline`` for normalisation +
    ordinal encoding + median imputation. All regularised models
    REQUIRE normalised features (otherwise the penalty is unevenly
    distributed across the coefficient vector).
    """
    loader = MLFPDataLoader()
    credit = loader.load("mlfp02", "sg_credit_scoring.parquet")

    target_col = (
        "credit_utilization" if "credit_utilization" in credit.columns else "income_sgd"
    )

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        data=credit,
        target=target_col,
        train_size=0.8,
        seed=SEED,
        normalize=True,
        categorical_encoding="ordinal",
        imputation_strategy="median",
    )

    feature_cols = [c for c in result.train_data.columns if c != target_col]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_cols,
        target_column=target_col,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_cols,
        target_column=target_col,
    )

    return X_train, y_train, X_test, y_test, col_info["feature_columns"]


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC 1D PROBLEM — for bias/variance and polynomial fits
# ════════════════════════════════════════════════════════════════════════


def make_sine_dataset(
    n: int = 100,
    noise_sigma: float = 0.2,
    seed: int = SEED,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]:
    """Generate a 1D noisy-sine regression problem.

    Returns:
        x_train_2d (n_train, 1), y_train (n_train,),
        x_test_2d (n_test, 1),   y_test  (n_test,),
        noise_variance (float, σ² — the irreducible error floor)

    The true function is ``y = sin(2πx) + ε`` with ε ~ N(0, σ²).
    The noise variance is returned so callers can use it in the
    bias-variance decomposition (σ² is the "irreducible noise" term).
    """
    rng = np.random.default_rng(seed)
    x = rng.uniform(0, 1, n)
    y = np.sin(2 * np.pi * x) + rng.normal(0, noise_sigma, n)

    split = int(n * 0.8)
    x_train = x[:split].reshape(-1, 1)
    x_test = x[split:].reshape(-1, 1)
    y_train = y[:split]
    y_test = y[split:]
    return x_train, y_train, x_test, y_test, noise_sigma**2


def make_poly_pipeline(degree: int) -> Pipeline:
    """Polynomial-features + scaler + linear-regression pipeline."""
    return Pipeline(
        [
            ("poly", PolynomialFeatures(degree, include_bias=False)),
            ("scaler", StandardScaler()),
            ("lr", LinearRegression()),
        ]
    )


# ════════════════════════════════════════════════════════════════════════
# REPORTING HELPERS
# ════════════════════════════════════════════════════════════════════════


def print_header(title: str) -> None:
    """Print a banner so each phase of a technique file is easy to spot."""
    print("\n" + "=" * 72)
    print(f"  {title}")
    print("=" * 72)


def format_results_table(rows: list[dict[str, Any]], cols: list[str]) -> str:
    """Render a small table of dicts as fixed-width text."""
    header = "  ".join(f"{c:>12}" for c in cols)
    sep = "-" * len(header)
    body = "\n".join(
        "  ".join(
            f"{row[c]:>12.4f}" if isinstance(row[c], float) else f"{row[c]:>12}"
            for c in cols
        )
        for row in rows
    )
    return f"{header}\n{sep}\n{body}"


def save_html_plot(fig: Any, name: str) -> Path:
    """Write a plotly figure into OUTPUT_DIR and return the path."""
    path = OUTPUT_DIR / name
    fig.write_html(str(path))
    return path




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 2.3: Lasso (L1) and ElasticNet
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Fit Lasso at many α values and read the sparsity pattern
#   - Explain L1 geometry: diamond constraint, corner solutions, exact zeros
#   - Use Lasso as built-in feature selection
#   - Blend L1 and L2 with ElasticNet for correlated features
#   - Draw the regularisation path and spot the "kinks" where L1 drops
#     features one by one
#   - Apply sparse selection to a Singapore insurance fraud scorecard
#
# PREREQUISITES:
#   - 02_ridge_regression.py (L2 geometry and Bayesian view)
#
# ESTIMATED TIME: ~40 minutes
#
# TASKS (5-phase R10):
#   1. Theory — the L1 diamond and why it produces zeros
#   2. Build — Lasso + ElasticNet fits across α and l1_ratio
#   3. Train — sparsity trajectory + ElasticNet sweep
#   4. Visualise — regularisation path (||β||₁ and non-zero count vs α)
#   5. Apply — AIA Singapore insurance fraud scorecard feature selection
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
from sklearn.linear_model import ElasticNet, Lasso
from sklearn.metrics import mean_squared_error

from kailash_ml import ModelVisualizer



## THEORY — L1 Regularisation


In [ ]:
# Lasso's objective replaces Ridge's squared-norm penalty with an
# absolute-value penalty:
#
#     min_β  ||y - Xβ||²  +  α · ||β||₁
#                             ─────────
#                             Sum of |β_i|
#
# GEOMETRY: ||β||₁ ≤ c traces a DIAMOND (a square rotated 45°). The
# MSE level sets are ellipses — they tend to first touch the diamond
# at a CORNER, and the corners of the diamond are precisely the points
# where some coordinate is zero. Result: Lasso zeroes out features
# exactly, performing built-in feature selection.
#
# NO CLOSED FORM: Because |x| is not differentiable at zero, there's no
# (X'X + αI)⁻¹ formula. Instead sklearn uses coordinate descent, which
# is why you'll see max_iter=10000 in production.
#
# BAYESIAN VIEW: L1 ⇔ Laplace prior P(β) ∝ exp(-|β|/b). The sharp peak
# at zero is what causes coordinates to snap to exactly zero in the MAP
# estimate.
#
# ELASTICNET: Sometimes you want "some sparsity" AND "group correlated
# features together". ElasticNet's objective is
#
#     min_β  ||y - Xβ||²  +  α · ( r·||β||₁ + (1-r)·||β||² )
#
# where r = l1_ratio ∈ [0,1]. r=1 is Lasso, r=0 is Ridge. Most real
# production scorecards use r≈0.5 to get robust sparsity on correlated
# features.



## TASK 2 — BUILD Lasso + ElasticNet fits


In [ ]:
print_header("Lasso & ElasticNet on Singapore Credit Data")

X_train, y_train, X_test, y_test, feature_names = load_credit_data()
print(f"Train: {X_train.shape}  Features: {len(feature_names)}")



## TASK 3 — TRAIN Lasso across the α sweep


In [ ]:
# Watch the sparsity column: as α grows, more coefficients are driven
# to exactly zero. That's L1 doing feature selection for you.

lasso_results: dict[float, dict[str, float]] = {}
for alpha in ALPHAS:
    lasso = Lasso(alpha=alpha, max_iter=10_000)
    lasso.fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, lasso.predict(X_train))
    test_mse = mean_squared_error(y_test, lasso.predict(X_test))
    l1_norm = float(np.sum(np.abs(lasso.coef_)))
    n_zero = int(np.sum(np.abs(lasso.coef_) < 1e-6))
    lasso_results[alpha] = {
        "train_mse": float(train_mse),
        "test_mse": float(test_mse),
        "l1_norm": l1_norm,
        "n_zero": n_zero,
        "coef": lasso.coef_.copy(),
    }

print(
    f"\n{'alpha':>10} {'train MSE':>12} {'test MSE':>12} "
    f"{'||β||₁':>10} {'zeros':>8}  sparsity"
)
print("-" * 62)
for alpha, r in lasso_results.items():
    pct = r["n_zero"] / len(feature_names) * 100
    print(
        f"{alpha:>10.3f} {r['train_mse']:>12.4f} {r['test_mse']:>12.4f} "
        f"{r['l1_norm']:>10.4f} {r['n_zero']:>8}  {pct:>5.0f}%"
    )

best_alpha_lasso, best_lasso_row = min(
    lasso_results.items(), key=lambda x: x[1]["test_mse"]
)
print(
    f"\nBest Lasso α = {best_alpha_lasso}  "
    f"(test MSE = {best_lasso_row['test_mse']:.4f}, "
    f"zeros = {best_lasso_row['n_zero']})"
)

# Show the surviving features at the best α
best_lasso_model = Lasso(alpha=best_alpha_lasso, max_iter=10_000).fit(X_train, y_train)
kept = sorted(
    [
        (name, coef)
        for name, coef in zip(feature_names, best_lasso_model.coef_)
        if abs(coef) > 1e-6
    ],
    key=lambda x: abs(x[1]),
    reverse=True,
)

print(f"\nLasso-selected features ({len(kept)}):")
for name, coef in kept[:10]:
    sign = "+" if coef > 0 else "-"
    print(f"  {sign} {name:<34} β = {coef:+.4f}")



### Checkpoint 1


In [ ]:
assert (
    lasso_results[100.0]["n_zero"] > lasso_results[0.001]["n_zero"]
), "Higher-α Lasso must zero out MORE coefficients"
print("\n[ok] Checkpoint 1 passed — Lasso sparsity increases with α")
# INTERPRETATION: Compare Lasso's surviving feature list against what
# a domain expert would pick. Lasso often agrees with expert intuition
# because strong signal overwhelms the L1 penalty, while weak /
# redundant features get zeroed first.



## TASK 3b — TRAIN ElasticNet across l1_ratio


In [ ]:
# At a fixed α we vary the mix between L1 and L2. At l1_ratio=0 we
# recover Ridge behaviour (no exact zeros). At l1_ratio=1 we recover
# Lasso (maximum sparsity). 0.5 is a common "all-round" choice.

print_header("ElasticNet — Mixing L1 and L2 (α fixed at 0.1)")
en_results: dict[float, dict[str, float]] = {}
for l1_ratio in [0.1, 0.3, 0.5, 0.7, 0.9]:
    en = ElasticNet(alpha=0.1, l1_ratio=l1_ratio, max_iter=10_000)
    en.fit(X_train, y_train)
    train_mse = mean_squared_error(y_train, en.predict(X_train))
    test_mse = mean_squared_error(y_test, en.predict(X_test))
    n_zero = int(np.sum(np.abs(en.coef_) < 1e-6))
    en_results[l1_ratio] = {
        "train_mse": float(train_mse),
        "test_mse": float(test_mse),
        "n_zero": n_zero,
    }

print(f"\n{'l1_ratio':>10} {'train MSE':>12} {'test MSE':>12} {'zeros':>8}")
print("-" * 46)
for l1_ratio, r in en_results.items():
    print(
        f"{l1_ratio:>10.1f} {r['train_mse']:>12.4f} {r['test_mse']:>12.4f} "
        f"{r['n_zero']:>8}"
    )



### Checkpoint 2


In [ ]:
assert (
    en_results[0.9]["n_zero"] >= en_results[0.1]["n_zero"]
), "l1_ratio=0.9 should zero at least as many coefficients as 0.1"
print("\n[ok] Checkpoint 2 passed — l1_ratio drives sparsity monotonically")
# INTERPRETATION: With CORRELATED features, pure Lasso arbitrarily picks
# one and drops the others — which is unstable. ElasticNet keeps
# correlated groups together with smaller shared coefficients.



## TASK 4 — VISUALISE the regularisation path


In [ ]:
# The reg-path plot is the single most informative visual in the whole
# exercise. Two curves:
#   1. Lasso ||β||₁ vs α — steps down, with visible "kinks" each time a
#      feature is dropped to zero
#   2. Lasso non-zero count vs α — monotonic staircase toward zero
#
# Save the figures to outputs/mlfp03_ex2_regularisation_cv/.

print_header("Regularisation Path Visualisation")
viz = ModelVisualizer()

fig_l1 = viz.training_history(
    {"||β||₁ (Lasso)": [lasso_results[a]["l1_norm"] for a in ALPHAS]},
    x_label="Regularisation Strength (α)",
)
fig_l1.update_layout(
    title="Lasso: L1 Norm of Coefficients vs α",
    xaxis_type="log",
)
path_l1 = save_html_plot(fig_l1, "lasso_l1_norm_path.html")

fig_sparse = viz.training_history(
    {
        "Non-zero coefs": [
            len(feature_names) - lasso_results[a]["n_zero"] for a in ALPHAS
        ]
    },
    x_label="Regularisation Strength (α)",
)
fig_sparse.update_layout(
    title="Lasso: Sparsity vs α — features dropping to zero",
    xaxis_type="log",
)
path_sparse = save_html_plot(fig_sparse, "lasso_sparsity_path.html")

print(f"\nSaved: {path_l1.name}")
print(f"Saved: {path_sparse.name}")



### Checkpoint 3


In [ ]:
coef_matrix = np.array([lasso_results[a]["coef"] for a in ALPHAS])
assert coef_matrix.shape == (
    len(ALPHAS),
    len(feature_names),
), "Coefficient matrix should be (n_alphas, n_features)"
print("\n[ok] Checkpoint 3 passed — regularisation path matrix shape correct")
# INTERPRETATION: Each kink in the path corresponds to a feature being
# zeroed out. This is the L1 diamond geometry in action — corners of the
# diamond are points where some coordinate is exactly zero.



## TASK 5 — APPLY: AIA Singapore Insurance Fraud Scorecard


Model                      | Features | Annual fraud savings | Compliance cost
---------------------------|----------|----------------------|-----------------
Legacy (hand-picked, 12)   |    12    |         (baseline)   |     S$72K
Lasso (best α, ~40)        |    40    |         +S$9.1M      |    S$240K
Unregularised OLS (240)    |   240    |         +S$9.4M      |    S$1.44M

Net: Lasso delivers ~97% of the fraud-detection uplift at ~17% of
the compliance cost of the unregularised model.


In [ ]:
# SCENARIO: AIA Singapore processes ~1.2M claims/year across life,
# health, and general insurance. The fraud team maintains a claim
# scoring model with 240 candidate features. Regulators (MAS + the
# General Insurance Association) require every production feature to
# be EXPLAINABLE — every β in the scorecard must have a sign-off memo.
#
# WHY LASSO IS THE RIGHT TOOL:
#   - 240 candidate features but only ~40 are "memo-able" without huge
#     documentation overhead. Lasso picks that subset automatically
#     instead of the fraud team running 240 univariate tests.
#   - Lasso's exact zeros mean the production scorecard can DROP
#     features entirely from the data pipeline — fewer upstream ETL
#     dependencies, fewer broken-ingest incidents.
#   - Sign-off memos scale linearly with feature count. 40 features at
#     1 day/memo = 40 days of compliance work; 240 features = 240 days.
#     Lasso saves ~200 compliance-days per annual model refresh.
#
# WHY ELASTICNET FOR A PRODUCTION REFRESH:
#   - Some features are highly correlated (e.g. "claim amount" vs "claim
#     amount relative to policy sum"). Pure Lasso randomly drops one,
#     creating unstable memos between refreshes. ElasticNet with r=0.5
#     keeps correlated groups together with shared smaller weights, so
#     the memo set is stable year on year.
#
# BUSINESS IMPACT (2026 AIA Singapore claims book, S$4.2B annual paid):
#   - Feature-selection compliance savings: ~200 analyst-days × S$1,800
#     = S$360K/year in reduced model governance cost.
#   - Fraud detection uplift from a focused 40-feature model: ~1.8
#     percentage points over a hand-picked 12-feature legacy scorecard,
#     worth ~S$9.1M/year in recovered/prevented fraudulent payouts
#     (assuming a 12% baseline fraud rate on the ~S$420M suspected
#     fraud pool).
#   - Operational savings from dropped ETL: each removed feature saves
#     ~S$4K/year in pipeline cost and one avoided incident per year.
#     200 dropped features ≈ S$800K in annual ETL cost avoided.

print_header("AIA Singapore Fraud Scorecard — Lasso Feature Selection")
print(
)



## REFLECTION


======================================================================
  WHAT YOU'VE MASTERED
======================================================================

  [x] Lasso objective: ||y-Xβ||² + α·||β||₁
  [x] Diamond geometry → corner solutions → exact zeros
  [x] Laplace prior as the Bayesian twin of L1
  [x] ElasticNet (α, l1_ratio) for correlated-feature stability
  [x] Regularisation path as a diagnostic visual
  [x] Using Lasso for governance-friendly feature selection at AIA

  KEY INSIGHT: Use Lasso when you believe "only a handful of features
  matter". Use Ridge when you believe "everything matters a little".
  Use ElasticNet when you're not sure AND correlated groups exist.

  NEXT: 04_cross_validation.py — we've been picking α by eyeball. Now
  we formalise it with nested CV, time-series CV, and GroupKFold.


In [ ]:
print(
)

